# AXI test 17: the serving side a fourth time, with one retry when the repair cannot fix a refused reply
**Why this test exists.** In the third run (test 16) the exact-state arm `S` held the right state on 88.3, 94.6 and 97.2 percent of its steps. Four of the six jobs that missed missed at a step where the gate **refused** the reply and the repair could not fix it, and the job was never asked again. The framework's own ladder has the next rung: **one retry**, with the framework's feedback, and then accept the refusal. This run puts that rung into the serving loop and **counts what it costs**. It runs the exact-state cells on the same jobs as before with the retry on (`R8`, `R32`, `R64`), and a pair on 64 jobs that no earlier run has seen, without the retry (`H0`) and with it (`H1`). The kept chat is not run. The predictions and the reading rules were written before this run in `docs/PROVIDER_SIDE_RETRY_TEST.md`.

Model: **Qwen2.5-7B-Instruct, 4-bit AWQ**, engine **vLLM 0.31.0**, cache fixed at **80,000 tokens**, GPU **T4**.

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. **Choose the T4 even if your plan offers a faster GPU:** every earlier number is from a T4, and another GPU would make the comparison with them unreliable. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. **Keep this browser tab open** while a cell runs.

**How long it takes (a guess from the third run):** installing the engine about 5 minutes, loading the model 5 to 10 minutes, the safety check 3 minutes, the five cells about 15 minutes. In all about 40 to 45 minutes.

**If Colab stops** (the tab closes, the GPU is taken away): reconnect, run Cells 1 to 5 again (do not forget Cell 4b), and then run the cell you were in. Finished cells are skipped; a cell that was cut off starts again from its beginning (the longest takes about 4 minutes).

Do not paste any keys anywhere.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: get the code from GitHub. If it stops with 'the code is not on GitHub yet', finish the update 62 commands in Codespaces (git push) and run this cell again

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/provider_side_retry.py"), "the code is not on GitHub yet: finish the update 62 commands (git push) first"
assert os.path.exists("/content/axi/docs/results/axi_provider_bounded.json"), "the third run's file is not on GitHub"
print("code ok")

### Cell 3: install the engine, version 0.31.0 (about 5 minutes). If Colab says 'Restart session', press it, then run Cells 2, 3, 4 and 5 again (Cell 3 will then be quick)

In [ ]:
!pip -q install vllm==0.31.0

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). If it prints 'Drive NOT connected', the run still works but nothing survives a disconnect

In [ ]:
import sys, os, json, time, glob, re
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_provider_retry")

### Cell 4b: fix the output stream (about 1 second). vLLM 0.31.0 asks the notebook for a file number of its output and a Colab cell has none, so Cell 5 stops with `io.UnsupportedOperation: fileno`. This cell gives the notebook's output the usual numbers 1 and 2. Run it once after every restart, before Cell 5

In [ ]:
import sys
for name, fd in (("stdout", 1), ("stderr", 2)):
    s = getattr(sys, name)
    try: s.fileno()
    except Exception: s.fileno = (lambda fd=fd: fd)
print("output numbers:", sys.stdout.fileno(), sys.stderr.fileno())

### Cell 5: load the model and the engine with the capped constraint (downloads about 5.5 GB once; about 5 to 10 minutes). It prints the engine's own numbers. If it fails with a red error, stop and send me the red text (if the red text says `fileno`, run Cell 4b first; if it says `StructuredOutputsParams`, send it to me and do nothing else)

In [ ]:
from axi.experiments import provider_side_retry as R
eng = R.BoundedVLLMEngine()
info = eng.info(); print(json.dumps({k: v for k, v in info.items() if k != "constraint"}, indent=1))
print("constraint:", info["constraint"]["kind"], "name at most", info["constraint"]["name_cap"], "characters", info["constraint"]["regex_sha256"][:16], "(the same for the first and the second request)")
eng.warm_up()
print("cache size the engine will use:", info["capacity_tokens"], "tokens (", info["capacity_source"], ")")

def go(label):
    """Runs one cell unless its file is already there; saves it and copies it to Drive. A cell that was cut off starts again."""
    name = f"axi_provider_r_{label}.json"; K, first, retry = R.CELLS[label]
    print(f"=== {label}: jobs {first} to {first + K - 1}, retry {'on' if retry else 'off'}: {store.restore(name)}")
    if os.path.exists(store.path(name)): print("   already finished, skipping"); return
    t0 = time.time()
    def progress(i): print(f"   round {i}   {time.time() - t0:6.0f} s", end="\r")
    c = R.run_cell(eng, label, progress=progress)
    R.save_cell(c, store.path(name)); store.persist(name)
    s = R.summarize_cell3(c); f = lambda x, d=1: "-" if x is None else f"{x:.{d}f}"
    print(f"\n   done: {s['steps_done']} steps, {s['wall_seconds']:.0f} s, {f(s['seconds_per_step'], 3)} s/step, asked again {s['retries']} times in {s['retry_rounds']} rounds ({f(s['retry_seconds'])} s), put right {s['retry_exact']}, exact steps {s['exact_share'] * 100:.1f} %, jobs exact to the end {s['jobs_exact_to_end']}, refused by the strict parser {s['unparsed_all']}, exact steps per hour {f(s['exact_per_hour'], 0)}")

### Cell 6: SAFETY CHECK (about 3 minutes). 3 jobs of 3 steps with the retry on, then ONE probe: the second request for step 12 of job 19, where the third run's engine answered `ADD[(3,0)->(3,0):4tworowsdown4col#4]` and the gate refused it. It prints the sizes, the cached tokens and what the engine answers when it is told why. **A wrong answer is not a STOP** (that is what the test is for). If it says STOP, do not run the next cells: send me the printout

In [ ]:
smoke = R.run_jobs(eng, [0, 1, 2], True, steps=3)
for x in smoke["sessions"][0]["log"]: print(f"step {x['t']}: prompt {x['n_prompt']:5d}  cached {x['n_cached']}  new {x['n_new']:3d}  {x['text'][:50]!r}")
allrep = [r for s in smoke["sessions"] for x in s["log"] for r in ([(x["first_parsed"], x["first_n_cached"])] if x.get("retried") else []) + [(x["parsed"], x["n_cached"])]]
ok_cached = all(c is not None for _, c in allrep)
ok_parse = all(p for p, _ in allrep)
probe = R.retry_probe(eng)
print(f"probe (job {probe['job']}, step {probe['step']}, {probe['kind']}): the first reply was {probe['first']!r}, refused by the gate ({probe['why']}); the engine is told {probe['feedback']!r} and answers {probe['reply']!r} ({probe['n_new']} tokens); the strict parser {'reads it' if probe['parsed'] else 'REFUSES it'}; the gate {'admits it' if probe['admitted'] else 'does not admit it'}; the oracle wrote {probe['oracle']!r}")
print()
print("STOP: this vLLM version does not report cached tokens. Send me the printout and the version above." if not ok_cached else
      "STOP: a reply of the safety cell was refused by the strict parser, so the constraint is not working. Send me the printout." if not ok_parse else
      "STOP: the probe's reply is refused by the strict parser. Send me the printout." if not probe["parsed"] else
      "STOP: the probe's reply used all 96 new tokens. Send me the printout." if probe["n_new"] >= 96 else
      "OK: cached counts are reported, every reply is well formed and the probe's reply ends. You can go on.")

### Cell 7: the five cells, in this order: `R8`, `R32`, `R64` (jobs 0 to 7, 0 to 31, 0 to 63, retry on), then `H0` and `H1` (jobs 64 to 127, retry off and on). About 15 minutes in all. A counter shows the round

In [ ]:
for label in R.ORDER: go(label)

### Cell 8: put the finished cells together, read the third run's file from the repository, print the table and the 12 reading-rule lines, and download your file. You can run it at any time, with whatever cells are finished

In [ ]:
from google.colab import files
THIRD = "/content/axi/docs/results/axi_provider_bounded.json"
paths = sorted(glob.glob(store.path("axi_provider_r_*.json")))
print(len(paths), "cells:", [os.path.basename(p) for p in paths])
m = R.merge_files(paths, store.path("axi_provider_retry.json"), third_run=THIRD); store.persist("axi_provider_retry.json")
old = R.load_third_run(THIRD)["cells"]
S = R.summarize_all(m["cells"]); R.print_cells(S); R.print_verdict(R.verdict_retry(m["cells"], old))
files.download(store.path("axi_provider_retry.json"))

### Done
File downloaded: `axi_provider_retry.json`. Upload it to the chat, with the cell files from your Drive folder `axi_provider_retry` (the `axi_provider_r_*.json` files), a screenshot of the printed table and of the 12 reading-rule lines, and the engine numbers printed by Cell 5 (vLLM version, GPU, cache size). Do not paste any keys anywhere.